In [1]:
'''
Assisted with Claude & various other repositories
'''

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import rasterio
import cv2
from torch.utils.data import Dataset, DataLoader
import random
from torchvision import transforms
import albumentations as A

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),  # Changed to BatchNorm2d
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),  # Added dropout
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),  # Changed to BatchNorm2d
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Calculate needed padding to match x2 dimensions
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, bilinear=False):
        super(UNet, self).__init__()
        self.n_channels = n_channels
        self.bilinear = bilinear

        # Added dropout rate
        self.dropout_rate = 0.1

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)

        # Added dropout before final layer
        self.dropout = nn.Dropout2d(self.dropout_rate)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)

        x = self.dropout(x)  # Added dropout

        b, c, h, w = x.shape
        x = x.permute(0, 2, 3, 1).contiguous()
        x = x.view(-1, c)
        x = self.fc(x)
        x = x.view(b, h, w, 1).permute(0, 3, 1, 2)
        return x


In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import cv2

class Normalize:
    """Normalization transform for each channel"""
    def __init__(self):
        # Define expected ranges for each channel
        self.ranges = {
            'Albedo.tif': (-1, 1),           # Typical albedo range
            'DEM.tif': (-32767, 3061),        # Approximate elevation range
            'Land_Cover.tif': (11, 250),      # Assuming land cover classes
            'NDVI.tif': (-1, 1),            # NDVI range
            'NDWI.tif': (-1, 1),            # NDWI range
            'LST.tif': (28, 175),             # Typical LST range in Fahrenheit
            'HeatIndex.tif': (1, 25)
        }

    def __call__(self, sample):
        x = sample['input']
        y = sample['target']

        # Normalize each channel
        for i, channel_name in enumerate(['Albedo', 'DEM', 'Land_Cover', 'NDVI', 'NDWI']):
            min_val, max_val = self.ranges[channel_name]
            x[i] = (x[i] - min_val) / (max_val - min_val)

        # Normalize target LST
        min_val, max_val = self.ranges['LST']
        y = (y - min_val) / (max_val - min_val)

        return {'input': x, 'target': y, 'mask': sample['mask']}

class SpatialAugmentation:
    """Spatial augmentation transform"""
    def __init__(self, p=0.5):
        self.augment = A.Compose([
            A.RandomRotate90(p=p),
            A.HorizontalFlip(p=p),
            A.VerticalFlip(p=p),
            A.GridDistortion(p=p/2),
        ])

    def __call__(self, sample):
        x = sample['input']
        y = sample['target']
        mask = sample['mask']

        # Apply same spatial transformation to input, target and mask
        augmented = self.augment(image=x.transpose(1, 2, 0),
                               mask=np.concatenate([y.transpose(1, 2, 0),
                                                  mask.transpose(1, 2, 0)], axis=-1))

        x = augmented['image'].transpose(2, 0, 1)
        y_mask = augmented['mask']
        y = y_mask[..., 0:1].transpose(2, 0, 1)
        mask = y_mask[..., 1:].transpose(2, 0, 1)

        return {'input': x, 'target': y, 'mask': mask}

class RasterDataset(Dataset):
    def __init__(self, file_list, transform=None, nodata_fill_value=-9999.0):
        self.file_list = file_list
        self.transform = transform
        self.nodata_fill_value = nodata_fill_value
        self.normalize = Normalize()
        self.spatial_aug = SpatialAugmentation()

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        sample_files = self.file_list[idx]

        # Process input channels
        channels = []
        for key in ['Albedo.tif', 'DEM.tif', 'Land_Cover.tif', 'NDVI.tif', 'NDWI.tif']:
            with rasterio.open(sample_files[key]) as src:
                channel = src.read(1).astype(np.float32)
            channel = np.where(np.isnan(channel), 0.0, channel)
            channels.append(channel)

        ref_shape = channels[0].shape
        fixed_channels = []
        for ch in channels:
            if ch.shape != ref_shape:
                ch_resized = cv2.resize(ch, (ref_shape[1], ref_shape[0]),
                                      interpolation=cv2.INTER_LINEAR)
                fixed_channels.append(ch_resized)
            else:
                fixed_channels.append(ch)

        x = np.stack(fixed_channels, axis=0)

        # Process target
        with rasterio.open(sample_files['LST.tif']) as src:
            y = src.read(1).astype(np.float32)

        valid_mask = ~np.isnan(y)
        valid_mask = valid_mask & (y != self.nodata_fill_value)
        y = np.where(valid_mask, y, 0.0)

        if y.shape != ref_shape:
            y = cv2.resize(y, (ref_shape[1], ref_shape[0]),
                         interpolation=cv2.INTER_LINEAR)
            valid_mask = cv2.resize(valid_mask.astype(np.uint8),
                                  (ref_shape[1], ref_shape[0]),
                                  interpolation=cv2.INTER_NEAREST)
            valid_mask = valid_mask.astype(bool)

        y = np.expand_dims(y, axis=0)
        valid_mask = np.expand_dims(valid_mask, axis=0)

        # Create sample and apply transforms
        sample = {'input': x, 'target': y, 'mask': valid_mask}

        # Apply normalization
        sample = self.normalize(sample)

        # Apply spatial augmentation during training
        if self.transform:
            sample = self.spatial_aug(sample)

        # Convert to tensors
        sample['input'] = torch.from_numpy(sample['input'])
        sample['target'] = torch.from_numpy(sample['target'])
        sample['mask'] = torch.from_numpy(sample['mask'])

        return sample


from tqdm import tqdm
import os
def list_files_in_folder(folder_path):
    files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if
             os.path.isfile(os.path.join(folder_path, f))]
    return files

def get_file_paths(folder_path):
    file_paths = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            full_path = os.path.abspath(os.path.join(root, file))
            file_paths.append(full_path)
    return file_paths
file_list = []
allAlbedoPixelFiles = []
for filePath in get_file_paths('./Data/X/less5CloudCover'):
    if 'Albedo' in filePath:
        allAlbedoPixelFiles.append(filePath)
for xPath in tqdm(allAlbedoPixelFiles, desc="Packing X,y to dictionary..."):
    fileParts = xPath.split('/')
    fileName, date, city, cloudCategory, dataType = fileParts[-1], fileParts[-2], fileParts[-3], fileParts[-4], fileParts[-5]
    sceneFiles = list_files_in_folder(os.path.dirname(os.path.abspath(xPath)))
    rasterDict = {}
    for rasterPath in sceneFiles:
        rasterName = rasterPath.split('/')[-1]
        rasterDict[rasterName] = rasterPath
        lstPath = xPath.replace('/X/', '/y/').replace('Albedo.tif', 'LST.tif')
        rasterDict['LST.tif'] = lstPath
    file_list.append(rasterDict)

import random
random.shuffle(file_list)
train_ratio = 0.8
train_size = int(len(file_list) * train_ratio)
train_file_list = file_list[:train_size]
test_file_list = file_list[train_size:]



Packing X,y to dictionary...: 100%|██████████| 7611/7611 [00:15<00:00, 481.31it/s]


In [3]:
class CombinedLoss(nn.Module):
    def __init__(self, mse_weight=1.0, mae_weight=1.0):
        super().__init__()
        self.mse_weight = mse_weight
        self.mae_weight = mae_weight
        self.mse = nn.MSELoss(reduction='none')
        self.mae = nn.L1Loss(reduction='none')

    def forward(self, pred, target, mask):
        # Ensure mask is boolean
        mask = mask.bool()

        # Calculate losses element-wise
        mse_loss = self.mse(pred, target)
        mae_loss = self.mae(pred, target)

        # Apply mask and take mean
        mse_loss = (mse_loss * mask).sum() / mask.sum()
        mae_loss = (mae_loss * mask).sum() / mask.sum()

        # Combine losses
        return self.mse_weight * mse_loss + self.mae_weight * mae_loss

def train_model(train_file_list, test_file_list, num_epochs=200):
    # Create datasets
    train_dataset = RasterDataset(train_file_list, transform=True)
    test_dataset = RasterDataset(test_file_list, transform=False)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

    # Setup device and model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet(n_channels=5, bilinear=False).to(device)

    # Setup loss and optimizer
    criterion = CombinedLoss(mse_weight=1.0, mae_weight=0.5).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                         factor=0.5, patience=5,
                                                         verbose=True)

    best_test_loss = float('inf')

    # Training loop
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for batch in tqdm(train_loader, desc="Training"):
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)
            mask = batch['mask'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            # Ensure shapes match
            if outputs.shape != targets.shape:
                outputs = F.interpolate(outputs, size=targets.shape[2:], mode='bilinear', align_corners=True)

            loss = criterion(outputs, targets, mask)
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Evaluation phase
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch in test_loader:
                inputs = batch['input'].to(device)
                targets = batch['target'].to(device)
                mask = batch['mask'].to(device)

                outputs = model(inputs)

                # Ensure shapes match
                if outputs.shape != targets.shape:
                    outputs = F.interpolate(outputs, size=targets.shape[2:], mode='bilinear', align_corners=True)

                loss = criterion(outputs, targets, mask)
                test_loss += loss.item()

        test_loss /= len(test_loader)

        # Update learning rate
        scheduler.step(test_loss)

        # Save best model
        if test_loss < best_test_loss:
            print("Saving best model based on test loss")
            best_test_loss = test_loss
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_test_loss': best_test_loss
            }, 'best_model.pth')

        print(f'Epoch [{epoch+1}/{num_epochs}] - '
              f'Train Loss: {train_loss:.4f} - Test Loss: {test_loss:.4f}')
    return model

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*The verbose parameter is deprecated.*"
)
model = train_model(train_file_list, test_file_list)

Training: 100%|██████████| 6088/6088 [19:23<00:00,  5.23it/s]


Saving best model based on test loss
Epoch [1/200] - Train Loss: 0.0982 - Test Loss: 0.0980


Training: 100%|██████████| 6088/6088 [19:15<00:00,  5.27it/s]


Epoch [2/200] - Train Loss: 0.0800 - Test Loss: 0.1021


Training: 100%|██████████| 6088/6088 [19:11<00:00,  5.29it/s]


...

Training: 100%|██████████| 6088/6088 [19:12<00:00,  5.28it/s]


Epoch [4/200] - Train Loss: 0.0676 - Test Loss: 0.0984


Training: 100%|██████████| 6088/6088 [19:11<00:00,  5.29it/s]


Epoch [5/200] - Train Loss: 0.0646 - Test Loss: 0.1111


Training: 100%|██████████| 6088/6088 [19:09<00:00,  5.29it/s]


In [5]:
import json
with open('test.json', 'w') as f:
    json.dump(test_file_list, f, indent=4)

In [6]:
import math
import torch
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model
model = UNet(n_channels=5, bilinear=False)
criterion = nn.MSELoss()
model.to(device)

# Load the checkpoint
checkpoint_path = "./best_model.pth"
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_dataset = RasterDataset(test_file_list)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

# Temperature range for denormalization
TEMP_MIN = -20
TEMP_MAX = 200
TEMP_RANGE = TEMP_MAX - TEMP_MIN

def denormalize_temp(normalized_values):
    """Convert normalized values back to Fahrenheit"""
    return normalized_values * TEMP_RANGE + TEMP_MIN

test_mse = 0.0
total_samples = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        inputs = batch['input'].to(device)
        targets = batch['target'].to(device)
        mask = batch['mask'].to(device)

        # Get model predictions
        outputs = model(inputs)

        # Denormalize both predictions and targets to Fahrenheit
        outputs_f = denormalize_temp(outputs)
        targets_f = denormalize_temp(targets)

        # Apply mask if available
        if mask is not None:
            outputs_f = outputs_f[mask]
            targets_f = targets_f[mask]

        # Calculate MSE in Fahrenheit
        mse = ((outputs_f - targets_f) ** 2).mean()

        # Accumulate error
        test_mse += mse.item() * inputs.size(0)
        total_samples += inputs.size(0)

# Calculate final RMSE in Fahrenheit
test_mse /= total_samples
test_rmse = math.sqrt(test_mse)

print(f"Test RMSE: {test_rmse:.2f}°F")

print("Accuracy:", str(test_rmse)) #67

Testing: 100%|██████████| 1523/1523 [01:46<00:00, 14.29it/s]

Test RMSE: 16.83°F
Accuracy: 16.82606232077001


In [8]:
def save_prediction_and_truth(model, test_loader, test_file_list, device):
    """
    Save both prediction and ground truth from test loader as georeferenced TIFFs
    """
    model.eval()

    with torch.no_grad():
        # Get one sample
        sample = next(iter(test_loader))
        inputs = sample['input'].to(device)
        targets = sample['target'].to(device)
        mask = sample['mask'].to(device)

        # Get corresponding LST file path
        lst_tif_path = test_file_list[0]['LST.tif']

        # Convert tensors to numpy arrays
        mask_np = mask.cpu().numpy().squeeze()
        targets_np = targets.cpu().numpy().squeeze()

        # Get model prediction
        outputs = model(inputs)
        predicted_np = outputs.cpu().numpy().squeeze()

        # Apply mask to both prediction and ground truth
        predicted_np[~mask_np] = np.nan
        targets_np[~mask_np] = np.nan

        # Get geospatial metadata from original LST file
        with rasterio.open(lst_tif_path) as src:
            profile = src.profile.copy()
            profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

            # Denormalize values back to Fahrenheit
            # Using the same range as in the Normalize class
            predicted_np = predicted_np * (250 - (-50)) + (-50)  # Updated range
            targets_np = targets_np * (250 - (-50)) + (-50)      # Updated range

            # Save prediction
            pred_filename = "predicted_LST.tif"
            with rasterio.open(pred_filename, "w", **profile) as dst:
                dst.write(predicted_np.astype(np.float32), 1)

            # Save ground truth
            truth_filename = "ground_truth_LST.tif"
            with rasterio.open(truth_filename, "w", **profile) as dst:
                dst.write(targets_np.astype(np.float32), 1)

            # Calculate and print some statistics for valid pixels
            valid_mask = ~np.isnan(predicted_np)
            if valid_mask.any():
                mae = np.mean(np.abs(predicted_np[valid_mask] - targets_np[valid_mask]))
                rmse = np.sqrt(np.mean((predicted_np[valid_mask] - targets_np[valid_mask])**2))
                print(f"Mean Absolute Error: {mae:.2f}°F")
                print(f"Root Mean Square Error: {rmse:.2f}°F")

        print(f"Saved files:")
        print(f"Predictions: {pred_filename}")
        print(f"Ground Truth: {truth_filename}")
        print(f"Original LST: {lst_tif_path}")


In [10]:
    # Assume model and test_loader are already defined
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load the trained model
    checkpoint = torch.load('best_model.pth')
    model = UNet(n_channels=5, bilinear=False).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])

    save_prediction_and_truth(model, test_loader, test_file_list, device)

Mean Absolute Error: 26.60°F
Root Mean Square Error: 27.01°F
Saved files:
Predictions: predicted_LST.tif
Ground Truth: ground_truth_LST.tif
Original LST: /work/ubh496/heat-island-test/Data/y/less5CloudCover/Denton_TX/2023-09/LST.tif


In [ ]:
import rasterio
import json

def get_tif_ranges(data_list):
    ranges = {
        'NDWI': {'min': float('inf'), 'max': float('-inf')},
        'LST': {'min': float('inf'), 'max': float('-inf')},
        'Land_Cover': {'min': float('inf'), 'max': float('-inf')},
        'NDVI': {'min': float('inf'), 'max': float('-inf')},
        'DEM': {'min': float('inf'), 'max': float('-inf')},
        'Albedo': {'min': float('inf'), 'max': float('-inf')}
    }

    for item in tqdm(data_list, desc="Getting Ranges"):
        for tif_type, path in item.items():
            with rasterio.open(path) as src:
                data = src.read(1)
                tif_name = tif_type.split('.')[0]
                ranges[tif_name]['min'] = min(ranges[tif_name]['min'], float(data.min()))
                ranges[tif_name]['max'] = max(ranges[tif_name]['max'], float(data.max()))

    return ranges

ranges = get_tif_ranges(file_list)
print(json.dumps(ranges, indent=4))

Getting Ranges: 100%|██████████| 7611/7611 [04:00<00:00, 31.67it/s]

{
    "NDWI": {
        "min": -1.0,
        "max": 1.0
    },
    "LST": {
        "min": 28.141216278076172,
        "max": 174.2246551513672
    },
    "Land_Cover": {
        "min": 11.0,
        "max": 250.0
    },
    "NDVI": {
        "min": -1.0,
        "max": 1.0
    },
    "DEM": {
        "min": -32767.0,
        "max": 3061.0
    },
    "Albedo": {
        "min": -0.017999999225139618,
        "max": 0.997999906539917
    }
}
